In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from numpy import linalg as LA

from scipy.integrate import solve_ivp
from scipy import integrate
from scipy.linalg import null_space

from itertools import chain
from scipy.integrate import solve_ivp

import pickle
import pandas as pd
import os

from concurrent.futures import ProcessPoolExecutor

In [ ]:
%matplotlib notebook

In [ ]:
# Upload the saved .py file
%run /home/katya-zats/Documents/Andreev_progr/Current_ABS_200.py

In [ ]:
current = sum_s_current(2, 0, 46) 
current_1 = sum_s_current(-2, 4, 46) 

print(current, current_1)

In [ ]:
# function that returns the momentum state (as string) for printing

def print_state(n):
    
    if len(bin(n)[:1:-1]) < N:
        new_bin = bin(n)[:1:-1]
        for i in range(N - len(bin(n)[:1:-1])):
            new_bin += '0'
            
        return new_bin
    else:
        return bin(n)[:1:-1]

In [ ]:
# the momentum vectors < q_i0 ... q_iNk | rho | q_j0 ... q_jNk >, where q's can be either 0 or 1 
# in case of fermions - by writing each i and j of the density matrix in the binary representation
# we get all possible combinations of occupied/non-occupied Nk momentum states

# this function finds whether the q'_iq entry (q = [0, Nk-1] index of the entry in the vector) is occupied or not 
def n(q, i):
    if q > len(bin(i)[:1:-1])-1:
        return 0
    else:
        return int(bin(i)[:1:-1][q])

In [ ]:
# # This bose distrib version works slower for large integrations but gives no overflow in exp warning

# def bose_distrib(x, T):
#     """ Bose distribution function that handles large x/T to avoid overflow in the exponent """  

#     y = x / T
#     cutoff = 500.0  # cutoff to avoid overflow in exp
    
#     if y >= 0:
#         if y > cutoff:
#             return 0.0  # for large y, n_B -> 0
#         else:
#             return 1.0 / (np.exp(y)-1)  
#     else:
#         y = -y
#         if y > cutoff:
#             return -1.0  # for large negative y, expression goes to -1 (since n_B -> 0)
#         else:
#             return -1.0 - 1.0 / (np.exp(y)-1)

In [ ]:
# This bose distrib function works faster for large integrations but gives an overflow in exp warning

def bose_distrib(x, T):
    
    if x>=0:
        n_B = 1/(np.exp(x/T)-1)
        return n_B
    else:
        n_B = 1/(np.exp(-x/T)-1)
        return -1-n_B

In [ ]:
# # This fermi distrib version works slower for large integrations but gives no overflow in exp warning

# def fermi_distrib(x, T):
#     """ Fermi distribution function that handles large x/T to avoid overflow in the exponent """    
    
#     y = x / T
#     cutoff = 500.0  # cutoff to avoid overflow in exp
    
#     if y > cutoff:
#         return 0.0  # for large positive y, 1/(exp(y)+1) -> 0
#     elif y < -cutoff:
#         return 1.0  # for large negative y, 1/(exp(y)+1) -> 1
#     else:
#         return 1.0 / (np.exp(y) + 1.0)

In [ ]:
# This fermi distrib version works faster for large integrations but gives an overflow in exp warning

def fermi_distrib(x, T):
    return 1/(np.exp(x/T)+1)

In [ ]:
# MBSs parameters

N = 4
N_state = 2**N

lambda_coupl = 0.1 # Andreev-continuum coupling rate
damping_param = 0.01 # 0.01
Omega = 0.01

T_ferm = 0.04
T_bos = 0.04

alpha_0 = 0.0001
omega_c = 1

In [ ]:
# spectral density function

def spectral_density(x, lambda_coupl, damping_param, Omega):
    
    J = lambda_coupl**2 * damping_param * (1/((x - Omega)**2 + damping_param**2/4) - \
                                                   1/((x + Omega)**2 + damping_param**2/4))
    
    J_ohm = alpha_0 * x * np.exp(-np.abs(x)/omega_c)
    
    return J + J_ohm

In [ ]:
plt.plot(phi_0/np.pi, roots[:,0])
plt.plot(phi_0/np.pi, roots[:,1])

plt.plot(phi_0/np.pi, roots[:,2])
plt.plot(phi_0/np.pi, roots[:,3])


plt.grid()
plt.xlabel(r'$\varphi$, $\pi$', fontsize=12)
plt.ylabel(r'$E_A$, $\Delta$', fontsize=12)

> ### Calculate transition rates:

In [ ]:
# G_nu_mu for phi_0 = pi

G_AL = np.zeros((len(phi_0), sol_n, sol_n), dtype = 'float64')

for i in range(len(phi_0)):
    for nu in range(sol_n):
        for mu in range(sol_n):
            omega_nu_mu = roots[i, nu] - roots[i, mu]

            G_AL[i, nu, mu] = I_AL_sq[i, mu, nu] * \
            spectral_density(omega_nu_mu, lambda_coupl, damping_param, Omega) * \
            (bose_distrib(omega_nu_mu, T_bos) + 1) 

            if math.isnan(G_AL[i, nu, mu]):
                G_AL[i, nu, mu] = 0

In [ ]:
# trans rates 0,1,2,3 -> 0,1,2,3
G_AL_pp = G_AL[:, 0:4, 0:4]

# trans rates 0,1,2,3 -> -0,-1,-2,-3
G_AL_pm = G_AL[:, 0:4, 4:8]

# trans rates -0,-1,-2,-3 -> 0,1,2,3
G_AL_mp = G_AL[:, 4:8, 0:4]

In [ ]:
def integr_gamma_in(epsilon, root_n, i, T_ferm, T_bos, lambda_coupl, damping_param, Omega):
    
    SC_DOS = np.abs(epsilon)/np.sqrt(epsilon**2-1)
    omega_p_lamd = epsilon - roots[i, root_n]
    
    I_lamd_p = np.real(sum_s_current(epsilon, root_n, i))
    
    J_spectr = spectral_density(omega_p_lamd, lambda_coupl, damping_param, Omega)
    
    bose = bose_distrib(omega_p_lamd, T_bos) + 1
    fermi_occ = fermi_distrib(epsilon, T_ferm)
    
    Gamma_in = SC_DOS * I_lamd_p * J_spectr * bose * fermi_occ
    
    return Gamma_in

In [ ]:
def integr_gamma_out(epsilon, root_n, i, T_ferm, T_bos, lambda_coupl, damping_param, Omega):
    
    SC_DOS = np.abs(epsilon)/np.sqrt(epsilon**2-1)
    omega_lamd_p = roots[i, root_n] - epsilon
    
    I_lamd_p = np.real(sum_s_current(epsilon, root_n, i))
    
    J_spectr = spectral_density(omega_lamd_p, lambda_coupl, damping_param, Omega)
    
    bose = bose_distrib(omega_lamd_p, T_bos) + 1
    fermi_empt = 1 - fermi_distrib(epsilon, T_ferm)
    
    Gamma_out = SC_DOS * I_lamd_p * J_spectr * bose * fermi_empt
    
    return Gamma_out

In [ ]:
def compute_integrals(i, lamd, T_bos, T_ferm, lambda_coupl, damping_param, Omega):
    
    Gamma_out_pl = integrate.quad(integr_gamma_out, 1.001, 100, args=(lamd, i,\
                    T_ferm, T_bos, lambda_coupl, damping_param, Omega), epsabs=1e-6, epsrel=1e-6)[0]
    
    Gamma_out_min = integrate.quad(integr_gamma_out, -100, -1.001, args=(lamd, i,\
                    T_ferm, T_bos, lambda_coupl, damping_param, Omega), epsabs=1e-6, epsrel=1e-6)[0]
    
    Gamma_in_pl = integrate.quad(integr_gamma_in, 1.001, 100, args=(lamd, i,\
                    T_ferm, T_bos, lambda_coupl, damping_param, Omega), epsabs=1e-6, epsrel=1e-6)[0]
    
    Gamma_in_min = integrate.quad(integr_gamma_in, -100, -1.001, args=(lamd, i,\
                    T_ferm, T_bos, lambda_coupl, damping_param, Omega), epsabs=1e-6, epsrel=1e-6)[0]
    
    # Return results as a tuple
    return (i, lamd, Gamma_out_pl, Gamma_out_min, Gamma_in_pl, Gamma_in_min)

#### Note:  
During the integration procedure he overflow in exponent warning will arise if choosing the faster bose/fermi distribution versions,  
choose versions avoiding the overflow (but due to if/else - slower) to switch off the warning.  
  
  
"IntegrationWarning: The occurrence of roundoff error is detected, which prevents the requested tolerance from being achieved" - this warning arises when the transition rates are integrated for the ABSs with $E > \Delta$, ($\Delta=1$). The integration procedure will give out NaN.  
This is a phenomenon of an ABS merging with continuum. Since the underlying theoretical approach considers only $|E|<\Delta$, the warning signals about the unexplored ground and can not be turned off by numerical adjustments. Needs a new theoretical approach to handle the ABSs merging into continuum.    
  
To avoid the IntegrationWarning and NaN outputs, adjust the parameters, so that ABSs energies lie below the SC gap. 

In [ ]:
Gamma_out_pl = np.zeros((len(phi_0), 4))
Gamma_out_min = np.zeros_like(Gamma_out_pl) 
Gamma_in_pl = np.zeros_like(Gamma_out_pl)
Gamma_in_min = np.zeros_like(Gamma_out_pl)

# Use ProcessPoolExecutor for parallel execution
with ProcessPoolExecutor() as executor:
    
    # Create a list of futures by submitting tasks for parallel execution
    futures = [executor.submit(compute_integrals, i, lamd, T_bos, T_ferm, lambda_coupl, damping_param, Omega)
               for i in range(len(phi_0)) for lamd in range(4)]
    
    # Collect results as they are completed
    for future in futures:
        i, lamd, out_pl, out_min, in_pl, in_min = future.result()
        
        # Store results in the appropriate arrays
        Gamma_out_pl[i, lamd] = out_pl
        Gamma_out_min[i, lamd] = out_min
        Gamma_in_pl[i, lamd] = in_pl
        Gamma_in_min[i, lamd] = in_min

In [ ]:
Gamma_in = Gamma_in_pl + Gamma_in_min
Gamma_out = Gamma_out_pl + Gamma_out_min

In [ ]:
plt.plot(phi_0/np.pi, Gamma_in)
plt.plot(phi_0/np.pi, Gamma_out)

In [ ]:
# set the NaN values from the integration (see Note), for the last ABS, to 0.

for i in range(len(phi_0)):
    for lamd in range(4):
        if math.isnan(Gamma_in[i, lamd]):
                Gamma_in[i, lamd] = 0
                Gamma_out[i, lamd] = 0    

In [ ]:
# Directory path
directory = '/home/katya-zats/Documents/Andreev_progr/'

# File name
file_name_in = 'Gamma_in_bx_4.pkl'
file_name_out = 'Gamma_out_bx_4.pkl'

# Full file path
file_path_in = os.path.join(directory, file_name_in)
file_path_out = os.path.join(directory, file_name_out)

In [ ]:
# download the arrays

# Open the file in binary write mode
with open(file_path_in, 'wb') as file:
    # Serialize and save the array using pickle
    pickle.dump(Gamma_in, file)
    
with open(file_path_out, 'wb') as file:
    # Serialize and save the array using pickle
    pickle.dump(Gamma_out, file)

In [ ]:
# upload the arrays

# Open the file in binary read mode
with open(file_path_in, 'rb') as file:
    # Load the array from the file using pickle
    Gamma_in = pickle.load(file)
    
with open(file_path_out, 'rb') as file:
    # Load the array from the file using pickle
    Gamma_out = pickle.load(file)

> ### Construct the steady state matrix

In [ ]:
lvl = np.array(range(N))
states = np.array(range(N_state))

n_v = np.vectorize(n)

nf_h = np.array([n_v(lvl, si) for si in states])

In [ ]:
Steady_M = np.zeros((len(phi_0), N_state, N_state), dtype='float64') 

for j in range(len(phi_0)):
    
    # loop over the raws - as many as there are states - 16
    for i in range(N_state):

        # loop over 4 levels
        for mu in range(4):

            n_mu = nf_h[i, mu]

            # G_in
            Steady_M[j, i, i + (-1)**n_mu * 2**mu] += n_mu * (Gamma_in[j, mu] )

            # G_out
            Steady_M[j, i, i + (-1)**n_mu * 2**mu] += (1-n_mu) * (Gamma_out[j, mu] )

            # G_in, G_out
            Steady_M[j, i, i] -= n_mu * (Gamma_out[j, mu]) + \
                                (1-n_mu) * (Gamma_in[j, mu] )

    #             loop over lambda' != lamda
            for nu in chain(range(mu), range(mu+1, 4)):

                n_nu = nf_h[i, nu]

                Steady_M[j, i, i + (-1)**n_mu * 2**mu + (-1)**n_nu * 2**nu] += \
                                n_mu * (1-n_nu) * G_AL_pp[j, nu, mu] + \
                                n_mu * n_nu * G_AL_mp[j, nu, mu] + \
                                (1-n_mu) * (1-n_nu) * G_AL_pm[j, nu, mu]

                Steady_M[j, i, i] -= (1-n_mu) * n_nu * G_AL_pp[j, nu, mu] + \
                                         (1-n_mu) * (1-n_nu) * G_AL_mp[j, nu, mu] +\
                                         n_mu * n_nu * G_AL_pm[j, nu, mu]


### Find occupation probabilities of the manybody AS in the steady state as the nullspace of Lindblad matrix Steady_M


In [ ]:
ns = np.zeros((len(phi_0), 16), dtype='float64')

for i in range(len(phi_0)):
    ns[i] = null_space(Steady_M[i], rcond=10e-17)[:,0]
    
# Normalizarion
steady_occup = ns
steady_occup /= np.sum(steady_occup, axis=1, keepdims=True)

In [ ]:
for i in range(N_state):
    print(nf_h[i], '\t', steady_occup[122, i])

In [ ]:
plt.plot(phi_0/np.pi, steady_occup[:,])

plt.grid()

In [ ]:
# File name
file_name_steady = 'steady_occup_200.pkl'

# Full file path
file_path_steady = os.path.join(directory, file_name_steady)

In [ ]:
# download steady occupations

# Open the file in binary write mode
with open(file_path_steady, 'wb') as file:
    # Serialize and save the array using pickle
    pickle.dump(steady_occup, file)

In [ ]:
# upload steady occupations

# Open the file in binary read mode
with open(file_path_steady, 'rb') as file:
    # Load the array from the file using pickle
    steady_occup = pickle.load(file)

### Dynamical parity stabilization

In [ ]:
# MBS index, for exchange
index = 12

# select phi_0 index
phi_ind = 84


# at t=0, the occuptaions of the MBS 0 and index will be exchanged
kick_steady = np.copy(steady_occup[phi_ind])
kick_steady[index] = steady_occup[phi_ind, 0] 
kick_steady[0] = steady_occup[phi_ind, index]

In [ ]:
# occupations at the chosen phase phi_ind

for j in range(N_state):
    plt.scatter(phi_0[phi_ind]/np.pi, steady_occup[phi_ind, j])
    
plt.grid()

In [ ]:
def rhs1(t, P):

    # find derivatives of N_state elements
    dP = np.zeros(N_state)
        
    dP = Steady_M[phi_ind] @ P
        
    return dP

In [ ]:
# Initial condition
P0 = kick_steady

# time evolution of the ocuupations after the kick at t=0
sol = solve_ivp(rhs1, (0.0, 10000000000.0), P0, method='LSODA', dense_output=True) 

ts = sol.t
ys = np.stack(sol.y)

In [ ]:
# plt.plot(ts, ys[8], color = 'brown', label = r'$P_{g,-}$')
# plt.plot(ts, ys[12], color = 'green', label = r'$P_{g,e}$')

plt.plot(ts, ys[0]+ys[3]+ys[5]+ys[6]+ys[9]+ys[10]+ys[12]+ys[15], label = r'$P_{even}$')
plt.plot(ts, ys[1]+ys[2]+ys[4]+ys[7]+ys[8]+ys[11]+ys[13]+ys[14], label = r'$P_{odd}$')

    
plt.legend(loc=1, bbox_to_anchor=(1, 1.))
plt.grid()
plt.xscale('log')
plt.xlabel("t", fontsize = 12)
plt.ylabel(r'$P_{\alpha, \beta}(t)$', fontsize = 15)

In [ ]:
data_sol = {'ts': ts,
            'even': ys[0]+ys[3]+ys[5]+ys[6]+ys[9]+ys[10]+ys[12]+ys[15],
            'odd': ys[1]+ys[2]+ys[4]+ys[7]+ys[8]+ys[11]+ys[13]+ys[14] }

output_df = pd.DataFrame(data_sol)

# # Save the DataFrame and arrays to a .dat file
output_df.to_csv('P_even+odd_15pid32_ge.dat', sep='\t', index=False)

In [ ]:
# # Create a dictionary containing the arrays
data_sol = {'ts': ts,
            '0000': ys[0],
            '1000': ys[1],
            '0100': ys[2],
            '1100': ys[3]}

# # Create a DataFrame from the dictionary
output_df = pd.DataFrame(data_sol)

# # Save the DataFrame and arrays to a .dat file
output_df.to_csv('P_SO_bz_eg.dat', sep='\t', index=False)